In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-4: PPO in depth

## Where this session fits

> **The limitation we start from: the Intro's policy was small and its reward free. What does a PPO run on a language model add?**

- **The question:** what does a PPO run on a language model add to the Intro's PPO, and where does it cost or break?
- **Position: session 4 of the RL for LLMs series (RLLM-1 to RLLM-14)**
    - RLLM-2 and RLLM-3 ended in an RL stage we treated as a box
    - this session opens the box
    - the next session, RLLM-5, is a case study that trains a reasoning model

Reward sources: people compare (RLHF) (RLLM-2); an AI compares (RLAIF) (RLLM-3); a program checks (RLVR) (RLLM-5) (this session opens the RL stage that consumes the reward) <!-- long-ok -->

- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**By the end you can**

1. name the networks in a PPO run on a language model, and say which are updated
2. say what the critic does for the update, and what it costs on long completions
3. compute per-token rewards, TD errors and GAE advantages for a short completion
4. compare PPO and plain GRPO piece by piece

**Intro sessions we use, and recall only briefly**

- B3 (a learned critic: TD error, GAE), B4 (ratio, clip), C1 (per-token reward, KL), C2 (GRPO)

**The plan of this session, in 7 steps**

1. **recall:** the PPO update, the token-level view, TD error, GAE, and the critic's two jobs
2. **the networks** of a PPO run on a language model
3. **one iteration:** why the batch goes stale, and the old policy
4. **the clip:** a brake on each token
5. **rewards and credit:** the per-token reward and a worked example
6. **the critic:** what it learns from, what it costs
7. **PPO and plain GRPO** side by side

Reading the signals of a run comes in RLLM-5.


## Step 1 of 7: recall, the PPO update in one line

**Why this section?** The rest of the session builds on three Intro results. This is a reminder, not a re-teach.

> **PPO takes several small steps on one batch, pushing each sampled token up or down by its advantage, with the push clipped.**

- **why a ratio:** a batch is costly to generate, so it is reused for several epochs
    - after the first step the policy is no longer the one that wrote it, and the ratio corrects for that
- the **ratio** of a sampled token: $r_{\tt}(\theta) = \pi_\theta(\text{token}_{\tt} \mid \text{prefix}_{\tt}) \,/\, \pi_{\text{old}}(\text{token}_{\tt} \mid \text{prefix}_{\tt})$
- the **advantage** $\hat{\advseq}_{\tt}$: how much better than expected the token turned out to be
    - positive: push the token's probability up; negative: push it down

$$L^{\text{clip}}(\theta) = \E_{\tt} \left[ \operatorname{min} \left( r_{\tt}(\theta)\, \hat{\advseq}_{\tt},\; \operatorname{clip}\left(r_{\tt}(\theta),\, 1-\epsilon_{\text{clip}},\, 1+\epsilon_{\text{clip}}\right)\, \hat{\advseq}_{\tt} \right) \right]$$

- $\epsilon_{\text{clip}}$ is typically $0.2$; $\pi_{\text{old}}$ is the policy that wrote the batch (B4)
- $\E_{\tt}$ is the average over the tokens of the batch

**Recall: TD error and GAE (B3)**

> **The critic's value of each prefix turns rewards into advantages in two lines: a TD error per token, then a backward sum.**

- the **TD error:** $\delta_{\tt} = \rew_{\tt} + \gamma\,\statevalfun_\phi(\text{next prefix}) - \statevalfun_\phi(\text{prefix})$
    - the reward plus the critic's value afterwards, minus its value before: the critic's surprise at token $\tt$
- **GAE**, run backward from the last token (the advantage after the last token is $0$):

$$\hat{\advseq}_{\tt} = \delta_{\tt} + \gamma \lambda \, \hat{\advseq}_{\tt+1}$$

- $\lambda = 0$ gives the TD error: low variance, but biased by the critic's errors
- $\lambda = 1$ gives the reward still to come minus the critic's value: no critic bias, but more variance
- $\lambda$ sets how far ahead a token's blame or praise looks (Step 5 shows it)

**Recall: the token-level view (RLLM-2), and the critic's two jobs**

<!-- seed-plant: critic-two-roles -->

> **PPO treats a completion as a game (state: the prefix, action: the next token), and its critic does two jobs, named here.**

- the reward model scores the whole completion, so its reward lands on the last token
- the KL term of Step 5 adds a small reward at every token
- the Intro used the critic for both jobs; this session names them
- **job 1, baseline:** subtracting the value of the prefix from the return lowers variance
- **job 2, credit assignment:** the value is defined for every prefix, so every token gets its own advantage
- **Keep this: job 2 returns in RLLM-5 and RLLM-6.**
    - RLLM-5 asks which rewards can give per-token credit
    - RLLM-6 meets a method that gives it up


## Step 2 of 7: the networks in a PPO run

**Why this section?** The first source of confusion on a language model is how many networks there are, and which ones learn.

> **Four networks matter. The policy and the critic are updated; the reference model and the reward model are frozen.**

<img src="images/rllm4_models.png" width=75%>

| Network | Input → output | Updated? | Role |
|:--|:--|:--|:--|
| **policy** $\pi_\theta$ | prefix → next-token distribution | yes | writes the completions |
| **critic** $\statevalfun_\phi$ | prefix → one number | yes | estimates the reward still to come: the baseline |
| **reference** $\pi_{\text{ref}}$ | prefix → next-token distribution | no | the SFT model: anchor of the KL reward (RLLM-2) |
| **reward model** | prompt and completion → one number | no | the reward at the last token (RLLM-2) |


## Step 3 of 7: one iteration, in order

**Why this section?** The schedule explains why some models are fresh and some are stale.

> **PPO alternates a costly generation phase with several cheap update epochs on the same batch.**

<img src="images/rllm4_loop.png" width=80%>

- **rollout:** generate completions from the current policy
    - usually the main cost, so the batch is reused
- **rewards:** one reward-model reward per completion, and a KL term for every token (C1)
- **advantages:** the critic values every prefix; GAE turns values and rewards into advantages
- **$K$ epochs:** $K$ passes over the same batch, in **mini-batches** (small slices)
    - each slice updates the policy (clipped) and the critic
- **refresh:** the updated policy samples the next batch

**Why the batch goes stale**

> **After the first mini-batch the policy is no longer the one that wrote the batch. The next step keeps the policy near it.**

- the **old policy** $\pi_{\text{old}}$ (B4) is the policy that generated the current batch
    - a snapshot, not a fifth network; its log-probabilities are stored during sampling
- supervised learning has fixed labels; here the data is the model's own earlier behavior
- two things go stale
    - the **samples** came from the old policy, and the ratio corrects for that only while the policies are close
    - the **critic** was fit to the policy of the last iteration
- the policy, the reference and the critic's body usually start from the SFT model (the critic's output layer is new): see Step 6


## Step 4 of 7: the clip, a brake on each token

**Why this section?** The batch goes stale (Step 3), so something must stop the policy from running away from it.

> **PPO stops pushing a token once its probability has moved more than $\epsilon_{\text{clip}}$ from the old policy's, in the direction the advantage asks.**

<img src="images/rllm4_clip.png" width=78%>

- left: a good token ($A > 0$) is pushed up until $r$ reaches $1 + \epsilon_{\text{clip}}$, then its gradient is zero
- right: a bad token ($A < 0$) is pushed down until $r$ reaches $1 - \epsilon_{\text{clip}}$
- the `min` keeps the pessimistic curve, so a token that moved the wrong way is still corrected
- with one update per batch ($K = 1$) the ratio is $1$ and the clip never acts: it matters only when a batch is reused
- the clip only zeroes the gradient of tokens already past the boundary: a token's ratio can still leave $[1-\epsilon_{\text{clip}}, 1+\epsilon_{\text{clip}}]$ within the $K$ epochs
- schematic, for one token

> **Keep this: the clip is a heuristic brake on each token, not a bound on the KL: our reading is that it gives no guarantee (the TRPO aside gives the argument). It returns in RLLM-6.**

<!-- seed-plant: clip-heuristic -->


## Step 5 of 7: the per-token reward

**Why this section?** In RLHF-style PPO every token gets a reward, though the reward model speaks only at the end.

> **The reward at token $\tt$ is the KL term $-\beta \log(\pi_\theta / \pi_{\text{ref}})$, plus the reward model's reward if $\tt$ is the last token.**

- this is Intro session C1's per-token reward
- it is computed once, at rollout, with the policy that sampled the batch ($\pi_\theta$ then), and held fixed during the $K$ epochs
- $\beta$ is the KL coefficient, the leash of RLLM-2
- the critic and GAE then treat it like any reward sequence
- the KL term is **sampled**: the log-ratio of the token that was drawn, not the full KL over the vocabulary
    - summed over a completion it estimates $\beta \, \KL(\pi_\theta \,\|\, \pi_{\text{ref}})$
    - a single token's term can be positive or negative (RLLM-2), and the example below has both
- plain GRPO puts the KL in the loss instead (C2). With $d = \log(\pi_\theta / \pi_{\text{ref}})$ as above, its estimator is $e^{-d} + d - 1$, never negative
    - with $r = \pi_{\text{ref}} / \pi_\theta$ this is $r - \log r - 1$, as in C2
    - it is a low-variance estimator of the KL used as a loss term, not the gradient of the KL
    - here the log-ratio is a reward, so its sign matters


## Step 5 of 7 (continued): worked example, credit for each token

**Why this section?** We follow one short completion through the machinery, with a critic and the KL reward.

> **With a critic, each token gets its own advantage, computed from the critic's surprise at each step.**

- a four-token completion with $\gamma = 1$ and $\beta = 0.1$; **made-up numbers, our illustration**
- drift exaggerated so the KL terms are visible (our illustration): a log-ratio of about $1.0$ per token is about $2.7\times$ the reference probability
- the reward model's reward is $0.2$
- before token 1 the critic expects $0.30$ in total
    - the completion earns $-0.05$: KL rewards $-0.25$, reward model $+0.2$
    - so it is well below its expectation, by $0.35$
- the value after the last token is $0$: the completion has ended
- the TD error and the GAE recursion are those of Step 1

| | token 1 | token 2 | token 3 | token 4 |
|:--|:--|:--|:--|:--|
| log-ratio $\log \pi_\theta / \pi_{\text{ref}}$ | 1.0 | -0.5 | 1.0 | 1.0 |
| KL reward $-\beta \times$ log-ratio | -0.10 | +0.05 | -0.10 | -0.10 |
| reward model's reward (last token only) | 0 | 0 | 0 | +0.20 |
| reward $\rew_\tt$ (the two rows above, added) | -0.10 | +0.05 | -0.10 | +0.10 |
| critic value $\statevalfun_\phi$ of the prefix before the token | 0.30 | 0.50 | 0.45 | 0.45 |
| TD error $\delta_\tt$ | +0.10 | 0.00 | -0.10 | -0.35 |
| advantage, $\lambda = 0.95$ | -0.29 | -0.41 | -0.43 | -0.35 |

<img src="images/rllm4_gae.png" width=95%>

- left: what the critic expected from each prefix, against the reward that actually came after it
- **before the next slide: does token 1 get praised or blamed at $\lambda = 0$?**
- **the choice of $\lambda$ changes who is blamed**
- $\lambda = 1$ gives *reward still to come minus the critic's value*: every token is told about the shortfall
    - tokens 2 and 3 are blamed most ($-0.45$)
    - the critic valued their prefixes at $0.50$ and $0.45$, and only $0.05$ and $0.00$ followed
- $\lambda = 0$ gives the TD error: token 1 is **praised** ($+0.10$), and only token 4, where the critic was surprised, takes the full $-0.35$
- $\lambda = 0.95$ lies between: token 1 gets $-0.29$, blamed through the later errors it leads to
- real completions are hundreds of tokens, so the choice matters more there
- **no critic, no choice:** GRPO gives all four tokens the same number (Step 7)


## Step 6 of 7: what the critic learns from

**Why this section?** The advantages above are only as good as the critic, so we need to know what it is trained to match.

> **The critic is trained by regression to a target built from this batch: the token's advantage plus the critic's own earlier value.**

- target $\hat R_{\tt} = \hat{\advseq}_{\tt} + \statevalfun_\phi(\text{prefix}_{\tt})$, with the values taken at rollout time and held fixed
- **value loss:** the mean squared gap between the critic's value and $\hat R_{\tt}$
    - below, token 1's value is pulled from $0.30$ toward $0.01$
- the TD-target idea of Intro session B3, run on every prefix of the batch, for $K$ epochs
- the policy and the critic have separate losses, updated on the same mini-batch

| | token 1 | token 2 | token 3 | token 4 |
|:--|:--|:--|:--|:--|
| advantage $\hat{\advseq}_\tt$, $\lambda = 0.95$ | -0.29 | -0.41 | -0.43 | -0.35 |
| critic's value before the update | 0.30 | 0.50 | 0.45 | 0.45 |
| **target** $\hat R_\tt$ = advantage + value | +0.01 | +0.09 | +0.02 | +0.10 |

- with $\lambda = 1$ the target is exactly the reward still to come
- with $\lambda < 1$ it blends in the critic's own next-prefix value


## Step 6 of 7 (continued): what the critic costs on a language model

**Why this section?** The critic's credit assignment is not free, and the price is the reason the next sessions look elsewhere.

> **The critic is a second network, starting with meaningless values, that must keep up with a policy that keeps moving.**

- **memory and compute:** its own parameters, gradients and optimizer state
    - a critic of the policy's size roughly doubles what is trained
    - a rough figure, not a measurement; a smaller critic costs less
- **the cold start:** the critic's output layer begins random, so early advantages are noisy
    - the critic is often the policy's network with a new output layer, or starts from the reward model (common in practice, our reading)
- **a critic that falls behind:** the policy changes, and the critic's targets change with it
    - a lagging critic gives noisy advantages, and the policy follows noise (our reading)
- **long completions (our reading):** one end reward is spread over hundreds of tokens, and an early prefix averages many continuations
    - known failure case: VC-PPO ([2503.01491](https://arxiv.org/abs/2503.01491)) and VAPO ([2504.05118](https://arxiv.org/abs/2504.05118)), not covered here


## Step 7 of 7: PPO and plain GRPO, side by side

**Why this section?** This is the trade at the heart of Intro session C2, piece by piece.

> **PPO buys per-token credit with a critic. Plain GRPO gives up per-token credit to remove the critic.**

| Piece of an update | PPO (this session) | Plain GRPO (C2) |
|:--|:--|:--|
| advantage (the weight) | one per token, from a critic (GAE) | one per completion, from the group |
| baseline | the critic's value | the group's mean reward |
| ratio and clip | per token, clip width $\epsilon_{\text{clip}}$ | per token, the same clip |
| KL penalty | a reward at every token | a term in the loss |

- **networks in training:** PPO trains two (policy, critic); plain GRPO trains one

> **In the worked example, PPO tells the four tokens apart; plain GRPO cannot.**

<img src="images/rllm4_credit.png" width=70%>

- real GRPO computes one number per completion from the group's rewards (C2), so it has no per-token bar here
- GRPO needs many completions per prompt, with varied rewards: it spends compute on them instead of on a critic


# What we established, and what it sets up

- **Established**
    - four networks: policy and critic learn; reference and reward model are frozen
    - one iteration: roll out, add the KL reward, compute advantages with the critic and GAE, update for $K$ epochs, refresh
    - the clip brakes a token once its ratio leaves $[1-\epsilon, 1+\epsilon]$: a heuristic, with no guarantee
    - the critic has two jobs, baseline and credit assignment, and costs a second network that must keep up
    - PPO and plain GRPO differ piece by piece: per-token credit against no critic
- **Sets up**
    - **the limitation:** the critic costs a network and is fragile on long completions
    - two separate choices are open: take the reward from a program instead of a learned model, and replace the critic by a group baseline
    - next session, RLLM-5, trains a reasoning model with a recipe that makes both choices
    - **reading a run** (reward, KL, value loss, clip fraction, entropy, and for GRPO all-equal groups and truncation) comes in RLLM-5


## Aside: where the clip comes from, TRPO

> **TRPO bounds the KL between old and new policy at every step. PPO's clip is a cheaper stand-in with no such bound.**

- the surrogate objective is exact at the old policy and drifts from the true improvement as the policy moves (Kakade and Langford 2002)
    - Intro session B4 measured the drift on small synthetic problems: it grows with the KL
- **TRPO** (Schulman et al. 2015) maximizes the surrogate subject to a KL bound

$$\max{\theta} \;\E_{\tt} \left[ r_{\tt}(\theta)\, \hat{\advseq}_{\tt} \right] \quad \text{subject to} \quad \E_{\state} \left[ \KL \left( \pi_{\theta_{\text{old}}}(\cdot \mid \state) \,\|\, \pi_\theta(\cdot \mid \state) \right) \right] \leq \delta_{\text{KL}}$$

    - **flag:** $\delta_{\text{KL}}$ is TRPO's bound, not the TD error $\delta_{\tt}$
- the theory behind the bound asks for steps so small that the practical TRPO uses the constraint form
- PPO's clip limits only the ratio of the *sampled* token: **our reading:** no bound on the KL, so a heuristic
